In [1]:
# pip install llama-index-vector-stores-chroma

In [2]:
# pip install llama-index-embeddings-openai 

In [3]:
import os
import chromadb
from dotenv import load_dotenv

load_dotenv()


from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    StorageContext, # lets wait
)

from llama_index.core.node_parser import SentenceSplitter
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)



In [4]:
docs = SimpleDirectoryReader(
    input_files=[
        "data_test/hr_policy_manual.pdf",
    ]
).load_data()

nodes = SentenceSplitter(chunk_size=512, chunk_overlap=64).get_nodes_from_documents(docs)



2026-10-25 21:07:26,492 - INFO - NumExpr defaulting to 10 threads.


In [5]:
chroma_client = chromadb.PersistentClient(path="./nb03_chroma")
col = chroma_client.create_collection("docs_v1")
vs = ChromaVectorStore(chroma_collection=col)




In [6]:
sc = StorageContext.from_defaults(vector_store=vs)
VectorStoreIndex(nodes, storage_context=sc)

2026-10-25 21:07:28,772 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [7]:
index = VectorStoreIndex.from_vector_store(vs, storage_context=sc)

In [8]:
query = "What is the WFH policy?"
retriever_basic = index.as_retriever(similarity_top_k=5)
retrieved_nodes = retriever_basic.retrieve(query)

2026-10-25 21:07:29,371 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [9]:
retrieved_nodes

[NodeWithScore(node=TextNode(id_='1fdf8af0-cb2c-4dcf-8366-8a8aa263baf9', embedding=None, metadata={'page_label': '1', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-06', 'last_modified_date': '2026-10-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='f4e5d2c2-c332-4188-967b-16d6c6f0f086', node_type='4', metadata={'page_label': '1', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-06', 'last_modified_date': '2026-10-24'}, hash='761800067c19f1a07c7b6b7bba214ed1dbbb7fb632691547daf09f2

In [10]:
from llama_index.core.vector_stores.types import MetadataFilters, ExactMatchFilter

In [11]:
query = "What is the leave policy?"

# 1. Create a filter that allows only chunks from the HR policy PDF
hr_filter = MetadataFilters(
    filters=[
        ExactMatchFilter(key="file_name", value="hr_policy_manual.pdf")
    ]
)


retriever_filtered = index.as_retriever(
    similarity_top_k=3,
    filters=hr_filter
)


In [12]:

retrieved_nodes = retriever_filtered.retrieve(query)
print(retrieved_nodes)

2026-10-25 21:07:29,677 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[NodeWithScore(node=TextNode(id_='92d561b4-428f-451b-b7b2-d501d624ea1d', embedding=None, metadata={'page_label': '2', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-06', 'last_modified_date': '2026-10-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='84d7a214-5339-4076-af5d-13da79df3e4d', node_type='4', metadata={'page_label': '2', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-06', 'last_modified_date': '2026-10-24'}, hash='31b824c645f91a258b1c7806fa7a2c4b6ce1f5dc0077a3746f1f996

In [13]:
from llama_index.core.query_engine import RetrieverQueryEngine
engine = RetrieverQueryEngine(retriever=retriever_filtered)

In [15]:
engine.query("What is the HR Policy Manual?")

2026-10-25 21:07:57,413 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-25 21:07:59,911 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response(response="The HR Policy Manual is a comprehensive document that outlines the employment terms, leave policies, compensation and benefits, and code of conduct for employees at NovaTech Solutions Pvt. Ltd. It serves as a guide for employees regarding their rights, responsibilities, and the company's policies, ensuring clarity and consistency in HR practices. The manual is confidential and is applicable for the fiscal year 2024-25.", source_nodes=[NodeWithScore(node=TextNode(id_='6d335767-bf5e-4d18-97c5-3ffe84011427', embedding=None, metadata={'page_label': '4', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-06', 'last_modified_date': '2026-10-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_m